## 7. Hidden SME Detection

Apply the final trained model to all **80 000 consumer cards** and produce a ranked list of hidden SME candidates.

Output format required by Mastercard MDQ:  
```
card_number, score
```
One row per consumer card. Higher score = higher probability the card belongs to a hidden business.


### 7.1 Score all consumer cards

In [ ]:
X_score = consumer_card_features[SELECTED_FEATURES].copy()

sme_probs = FINAL_MODEL.predict_proba(X_score)[:, 1]

predictions = pd.DataFrame({
    'card_number': consumer_card_features['card_number'].values,
    'sme_score':   sme_probs
}).sort_values('sme_score', ascending=False).reset_index(drop=True)

print('Scored consumer cards:', len(predictions))
print('Score range: ', round(predictions['sme_score'].min(), 4),
      '-', round(predictions['sme_score'].max(), 4))
display(predictions.head(10))

### 7.2 SME potential score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(predictions['sme_score'], bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('SME Score')
axes[0].set_ylabel('Number of cards')
axes[0].set_title('Distribution of SME Scores — All Consumer Cards')

# Zoom into high-score tail (potential hidden SMEs)
axes[1].hist(predictions.loc[predictions['sme_score'] > 0.5, 'sme_score'],
             bins=50, color='tomato', edgecolor='none')
axes[1].set_xlabel('SME Score')
axes[1].set_ylabel('Number of cards')
axes[1].set_title('High-Score Tail (score > 0.5)')

plt.tight_layout()
plt.show()

for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    n = (predictions['sme_score'] >= thr).sum()
    pct = n / len(predictions) * 100
    print(f'Score >= {thr:.2f}: {n:5d} cards  ({pct:.2f}%)')

### 7.3 Segment hidden SME candidates

Three tiers based on score:

| Tier | Score | Interpretation |
|---|---|---|
| High potential | ≥ 0.80 | Very likely hidden business — prioritize for review |
| Medium potential | 0.60 – 0.80 | Likely atypical consumer — worth investigation |
| Low potential | < 0.60 | Consumer-like behaviour |


In [ ]:
def assign_tier(score):
    if score >= 0.80:
        return 'High potential'
    elif score >= 0.60:
        return 'Medium potential'
    else:
        return 'Low potential'

predictions['tier'] = predictions['sme_score'].apply(assign_tier)

tier_counts = predictions['tier'].value_counts()
print(tier_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors_tier = {'High potential': '#e74c3c',
               'Medium potential': '#f39c12',
               'Low potential': '#2ecc71'}
for tier, cnt in tier_counts.items():
    ax.bar(tier, cnt, color=colors_tier.get(tier, 'gray'))
ax.set_ylabel('Number of cards')
ax.set_title('Hidden SME Candidate Segments')
plt.tight_layout()
plt.show()

### 7.4 Lead ranking — top 100 hidden SME candidates

In [ ]:
top100 = predictions.head(100).copy()
top100.index = range(1, 101)
top100.index.name = 'rank'

display(top100.head(20))

# Enrich with key behavioral features for manual inspection
top100_enriched = top100.merge(
    consumer_card_features[['card_number'] + SELECTED_FEATURES[:10]],
    on='card_number',
    how='left'
)
display(top100_enriched.head(10))

### 7.5 Behavioral profile — high potential vs all consumers

Compare key feature medians between the high-potential SME segment and the rest.  
This validates that the model is picking up real business-like signals.


In [ ]:
high_pot_cards = predictions.loc[predictions['tier'] == 'High potential', 'card_number']
high_pot_feats = consumer_card_features[
    consumer_card_features['card_number'].isin(high_pot_cards)
][SELECTED_FEATURES]

all_con_feats = consumer_card_features[SELECTED_FEATURES]

signal_features = [
    'total_amount', 'mean_amount', 'b2b_txn_share', 'b2b_amount_share',
    'recurring_share', 'repeated_amount_share', 'top_merchant_share',
    'merchant_hhi', 'foreign_share', 'us_share', 'night_share',
    'weekend_share', 'large_txn_share'
]
signal_features = [f for f in signal_features if f in SELECTED_FEATURES]

profile = pd.DataFrame({
    'feature':           signal_features,
    'high_potential_med': [high_pot_feats[c].median() for c in signal_features],
    'all_consumer_med':  [all_con_feats[c].median()   for c in signal_features],
})
profile['ratio'] = (profile['high_potential_med'] /
                    profile['all_consumer_med'].replace(0, np.nan)).round(2)
display(profile.round(4))

### 7.6 Final predictions — save submission file

In [ ]:
final_submission = predictions[['card_number', 'sme_score']].copy()
final_submission.columns = ['card_number', 'score']

final_submission.to_csv('final_predictions.csv', index=False)
print('Saved: final_predictions.csv')
print('Shape:', final_submission.shape)
display(final_submission.head(10))

### 7.7 Save final model

In [ ]:
joblib.dump(FINAL_MODEL,      'final_model.pkl')
joblib.dump(SELECTED_FEATURES,'selected_features.pkl')
print(f'Saved: final_model.pkl ({FINAL_MODEL_NAME})')
print('Saved: selected_features.pkl')
print(f'Number of features: {len(SELECTED_FEATURES)}')

### 7.8 Summary

The pipeline produces a scored ranking of all 80 000 consumer cards by their probability of being a hidden SME.  
Key results:
- **Final model:** CatBoost or XGBoost (chosen by test AUC)
- **Submission file:** `final_predictions.csv` — `card_number, score` for all consumer cards
- **High-potential SME segment:** cards with score ≥ 0.80
- **Evaluation metric:** AUC-ROC (as required by Mastercard MDQ)

The top-ranked cards consistently show business-like behavioral patterns:  
high B2B MCC share, recurring payments, high transaction amounts, low weekend activity, and US/Ireland foreign spend patterns.
